In [63]:
import pandas as pd
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from src.data_loader import load_raw_churn_data, save_csv
from src.config import CHURN_CLEAN_FILE

# Customer churn data cleaning

This notebook prepares the raw customer churn dataset for analysis. The cleaning steps focus on fixing data types, handling missing values, simplifying service-related categories, encoding binary variables, and removing non-analytical identifier fields.

In [64]:
df = load_raw_churn_data()


In [65]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [67]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [68]:
(df['TotalCharges'] == ' ').sum()

np.int64(11)

In [69]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.isna().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [70]:
df = df.dropna()
df.shape

(7032, 21)

# Missing values and data types

Initial checks suggested that the dataset had no missing values, but a closer review showed that `TotalCharges` was stored as an object instead of a numeric column. This happened because 11 rows contained blank spaces rather than valid numeric values.

To fix this, `TotalCharges` was converted to numeric using coercion, and the resulting missing rows were dropped.

# Duplicate check

The dataset was checked for duplicate records before further transformations.

In [71]:
df.duplicated().sum()

np.int64(0)

# Category cleanup

# Standardizing service-related categories

Several service columns contain the category `'No internet service'`. In these fields, that value overlaps with `'No'` because the customer does not use the service in either case.

To simplify the dataset and make the variables easier to analyze, `'No internet service'` was standardized to `'No'` before binary encoding.

In [72]:
internet_service_cols = [
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
]

df[internet_service_cols] = (
    df[internet_service_cols]
    .replace('No internet service', 'No')
    .apply(lambda x: x.map({'Yes': 1, 'No': 0}))
)

In [73]:
yes_no_col = [
    'MultipleLines',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling',
    'Churn',
]

In [74]:
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

df[yes_no_col] = df[yes_no_col].replace({'Yes': 1, 'No': 0})

/var/folders/0m/gbwrf68d7x1d396vdkq6b3940000gn/T/ipykernel_59189/1330546205.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[yes_no_col] = df[yes_no_col].replace({'Yes': 1, 'No': 0})


# Binary encoding

Binary customer and service columns were converted from `'Yes'` / `'No'` to `1` / `0` to make the dataset easier to analyze.

For `MultipleLines`, the value `'No phone service'` was first standardized to `'No'` before encoding.

# Removing identifier column

The `customerID` column was removed because it is only a unique identifier and does not add analytical value.

In [75]:
df = df.drop(columns='customerID')

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   object 
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   int64  
 3   Dependents        7032 non-null   int64  
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   int64  
 6   MultipleLines     7032 non-null   int64  
 7   InternetService   7032 non-null   object 
 8   OnlineSecurity    7032 non-null   int64  
 9   OnlineBackup      7032 non-null   int64  
 10  DeviceProtection  7032 non-null   int64  
 11  TechSupport       7032 non-null   int64  
 12  StreamingTV       7032 non-null   int64  
 13  StreamingMovies   7032 non-null   int64  
 14  Contract          7032 non-null   object 
 15  PaperlessBilling  7032 non-null   int64  
 16  PaymentMethod     7032 non-null   object 
 17  

# Save cleaned dataset

The cleaned dataset is saved for use in the exploratory analysis notebook.

In [77]:
save_csv(df, CHURN_CLEAN_FILE)